# Descargar librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ventanas_mejoras import expand_to_windows, build_extra_features, category_burst_report

# Leer y separar archivo

In [2]:
df = pd.read_csv('../data/logs.csv')
df = df.drop(columns=['_ignored'])
df_llm = df[df['_index'].str.startswith('llm-logs', na=False)]
df_llm = df_llm.drop(columns=['http_status_code', 'service_id', 'client_ip'])
df_system = df[df['_index'].str.startswith('sap-logs', na=False)]
df_system = df_system.drop(columns=['llm_error_message', 'llm_response_size_bytes', 'sap_llm_response_size', 'sap_llm_response_time', 'llm_prompt_tokens', 'llm_completion_tokens', 'llm_total_tokens', 'llm_response_time_ms', 'llm_model_id', 'llm_status', 'llm_cost_usd', 'llm_prompt_category', 'llm_temperature', 'llm_top_p', 'llm_stream', 'llm_finish_reason', 'llm_prompt_id', 'llm_provider', 'llm_prompt'])

# System

In [ ]:
df_system.columns

In [ ]:
lista_fila = df_system.iloc[0].tolist()
display(lista_fila)

## Variables de las ventanas (system)

Cada ventana dura **5 minutos** y avanza cada **2.5 minutos**, por lo que cada log pertenece a **dos ventanas consecutivas** (traslape del 50%). Solo se conservan ventanas **completas**: se descartan la primera y la última del periodo, que tienen la mitad de los datos y distorsionan la detección.

### 1. Volumen y ritmo
- **total_requests**: número de logs en la ventana.
- **request_rate_std**: desviación estándar, en segundos, del tiempo entre solicitudes consecutivas dentro de la ventana. Se calcula con `@timestamp`, que tiene resolución de 1 segundo.
- **interval_cv**: coeficiente de variación (desviación estándar / promedio) del tiempo entre solicitudes, calculado con `@event_time_requested`, que tiene precisión de milisegundos. Valores altos indican tráfico en ráfagas; valores bajos, tráfico regular. Es independiente del volumen.

### 2. Origen del tráfico (IPs)
- **unique_ips_count**: cantidad de IPs distintas (`client_ip`).
- **ip_entropy**: entropía de Shannon normalizada de las solicitudes por IP. Vale 1 si todas las IPs generan lo mismo y baja hacia 0 si una IP concentra el tráfico.
- **ip_top_share**: fracción de las solicitudes generada por la IP más activa de la ventana. Complementa a `ip_entropy`.
- **novel_ips**: número de IPs vistas por primera vez en toda la base dentro de esta ventana. Las primeras 24 horas se toman como calentamiento y valen 0.

### 3. Respuestas HTTP
- **error_rate_400** y **error_rate_500**: proporción de códigos 4xx y 5xx sobre los códigos HTTP distintos de 200. El resto corresponde a otros códigos (p. ej. 3xx). Son composición entre los no-200, no tasa de fallos.
- **non_200_rate**: proporción de solicitudes con código distinto de 200 sobre el **total** de la ventana. Detecta aumentos de fallos que `error_rate_*` no muestra cuando la mezcla 4xx/5xx se mantiene.

### 4. Tipo de log (`sap_function_log_type`)
- **ratio_error**, **ratio_warn**, **ratio_security**, **ratio_info**, **ratio_audit**: proporción de logs de cada tipo sobre los logs con tipo asignado.
- `ratio_security` agrupa mensajes de seguridad de bajo riesgo, con redacción tranquilizadora (p. ej. "within limit", "no action taken"). `ratio_audit` incluye los eventos de auditoría más sensibles (intentos de escalamiento de privilegios, accesos no autorizados bloqueados, descargas masivas de datos), por lo que es el mejor indicador de actividad de seguridad entre los ratios.

### 5. Rutas y métodos
- **get_vs_post_ratio**: proporción de solicitudes GET sobre el total (el resto son POST u otros métodos).
- **novel_paths**: número de rutas (`heathers_request_path`) vistas por primera vez en la base dentro de esta ventana. Las primeras 24 horas valen 0.

### 6. Desviación de la mezcla de categorías (`js_*`)
Para cada columna categórica se calcula la **divergencia de Jensen-Shannon** (en bits, entre 0 y 1) entre la distribución de categorías de la ventana y la distribución global. 0 significa que la ventana es idéntica a lo habitual; sube cuando aparecen categorías nuevas o cambian las proporciones. Su nivel base depende del tamaño de la ventana y del número de categorías, por lo que se comparan ventanas entre sí y no columnas entre sí.
- **js_http_status_code**, **js_headers_http_request_method**, **js_sap_function_log_type**, **js_heathers_request_path**, **js_sap_function_application**, **js_sap_source_type**, **js_sap_app_env**, **js_region_code**, **js_headers_content_type**.
- `headers_http_host` se omite porque es equivalente 1:1 a `region_code`. `sap_function_message` no se incluye por ser texto libre.

### Variables descartadas
- **request_rate_avg**: el promedio de intervalos entre solicitudes es ≈ 300 / total_requests (correlación 0.9987), es decir, repite el volumen.
- **unique_paths_ratio**: hay ~67 rutas distintas por ventana, así que equivale a ≈ 67 / total_requests. Se reemplaza por `novel_paths` y `js_heathers_request_path`.

## Generar ventanas

In [ ]:
window_size = pd.Timedelta(minutes=5)
window_step = pd.Timedelta(minutes=2.5)

# Se usa @timestamp como tiempo del evento para construir las ventanas.
required_columns = [
    '@timestamp', 'http_status_code', 'client_ip', 'sap_function_log_type',
    'heathers_request_path', 'headers_http_request_method',
    # nuevas
    '@event_time_requested', 'sap_function_application', 'sap_source_type',
    'sap_app_env', 'region_code', 'headers_http_host', 'headers_content_type',
]

logs = df_system[required_columns].copy()
logs['timestamp'] = pd.to_datetime(logs['@timestamp'], utc=True, errors='coerce')

if logs['timestamp'].isna().any():
    invalid_count = logs['timestamp'].isna().sum()
    raise ValueError(f'Hay {invalid_count} logs sin un @timestamp válido.')

logs['http_status_code'] = pd.to_numeric(logs['http_status_code'], errors='coerce')
logs['log_type'] = logs['sap_function_log_type'].astype('string').str.upper()
logs['method'] = logs['headers_http_request_method'].astype('string').str.upper()

events = expand_to_windows(logs)

status = events['http_status_code']
log_type = events['log_type']
events['is_non_200'] = status.notna() & status.ne(200)
events['is_4xx'] = status.between(400, 499).fillna(False)
events['is_5xx'] = status.between(500, 599).fillna(False)
events['has_log_type'] = log_type.notna()
events['is_error'] = log_type.eq('ERROR').fillna(False)
events['is_warning'] = log_type.eq('WARNING').fillna(False)
events['is_security'] = log_type.eq('SECURITY').fillna(False)
events['is_audit'] = log_type.eq('AUDIT').fillna(False)
events['is_info'] = log_type.eq('INFO').fillna(False)
events['is_get'] = events['method'].eq('GET').fillna(False)

grouped = events.groupby('window_start', sort=True)
df_batch_system = grouped.agg(
    total_requests=('timestamp', 'size'),
    unique_ips_count=('client_ip', 'nunique'),
    unique_paths_count=('heathers_request_path', 'nunique'),
    non_200_count=('is_non_200', 'sum'),
    count_4xx=('is_4xx', 'sum'),
    count_5xx=('is_5xx', 'sum'),
    log_type_count=('has_log_type', 'sum'),
    count_error=('is_error', 'sum'),
    count_warn=('is_warning', 'sum'),
    count_security=('is_security', 'sum'),
    count_info=('is_info', 'sum'),
    count_audit=('is_audit', 'sum'),
    count_get=('is_get', 'sum'),
)

# Los intervalos entre solicitudes consecutivas se calculan dentro de cada ventana, en segundos.
events_sorted = events.sort_values(['window_start', 'timestamp'], kind='mergesort')
intervals = events_sorted.groupby('window_start', sort=False)['timestamp'].diff().dt.total_seconds()
interval_stats = intervals.groupby(events_sorted['window_start']).agg(['mean', 'std'])
df_batch_system['request_rate_avg'] = interval_stats['mean']
df_batch_system['request_rate_std'] = interval_stats['std']

# Tasas como fracciones [0, 1]; el denominador HTTP incluye códigos distintos de 200.
non_200_denominator = df_batch_system['non_200_count'].replace(0, np.nan)
df_batch_system['error_rate_400'] = df_batch_system['count_4xx'] / non_200_denominator
df_batch_system['error_rate_500'] = df_batch_system['count_5xx'] / non_200_denominator
log_type_denominator = df_batch_system['log_type_count'].replace(0, np.nan)
df_batch_system['ratio_error'] = df_batch_system['count_error'] / log_type_denominator
df_batch_system['ratio_warn'] = df_batch_system['count_warn'] / log_type_denominator
df_batch_system['ratio_security'] = df_batch_system['count_security'] / log_type_denominator
df_batch_system['ratio_info'] = df_batch_system['count_info'] / log_type_denominator
df_batch_system['ratio_audit'] = df_batch_system['count_audit'] / log_type_denominator
df_batch_system['unique_paths_ratio'] = (
    df_batch_system['unique_paths_count'] / df_batch_system['total_requests']
)
df_batch_system['get_vs_post_ratio'] = (
    df_batch_system['count_get'] / df_batch_system['total_requests']
)

# Entropía de Shannon normalizada: 0 = una IP; 1 = solicitudes uniformes entre IPs.
ip_counts = (
    events.dropna(subset=['client_ip'])
    .groupby(['window_start', 'client_ip'], observed=True)
    .size()
)
ip_probabilities = ip_counts / ip_counts.groupby(level=0).transform('sum')
shannon_entropy = -(ip_probabilities * np.log2(ip_probabilities)).groupby(level=0).sum()
unique_ips_per_window = ip_counts.groupby(level=0).size()
normalized_entropy = shannon_entropy.div(np.log2(unique_ips_per_window)).where(
    unique_ips_per_window > 1, 0
)
df_batch_system['ip_entropy'] = normalized_entropy.reindex(df_batch_system.index).fillna(0)

# El índice identifica el inicio de ventana; se eliminan contadores intermedios.
df_batch_system = df_batch_system.drop(
    columns=[
        'unique_paths_count', 'non_200_count', 'count_4xx', 'count_5xx',
        'log_type_count', 'count_error', 'count_warn', 'count_security',
        'count_info', 'count_audit', 'count_get',
    ]
)
df_batch_system.index.name = 'window_start'
df_batch_system = df_batch_system[
    [
        'total_requests', 'request_rate_std', 'request_rate_avg',
        'unique_ips_count', 'error_rate_400', 'error_rate_500',
        'ratio_error', 'ratio_warn', 'ratio_security', 'ratio_info',
        'ratio_audit', 'ip_entropy', 'unique_paths_ratio', 'get_vs_post_ratio',
    ]
]

extra = build_extra_features(events)
df_batch_system = df_batch_system.join(extra, how='inner')
df_batch_system = df_batch_system.drop(columns=['request_rate_avg', 'unique_paths_ratio'])

display(df_batch_system.head())

In [ ]:
category_burst_report(logs)

In [ ]:
from ventanas_mejoras import check_unused_columns
res = check_unused_columns(df_system)
for k, v in res.items():
    print('##', k); print(v)

In [ ]:
from ventanas_mejoras import message_family_report
mf = message_family_report(df_system)
print(mf['n_familias'])
print(mf['frecuencia_familias'])
print(mf['familias_raras_x_log_type'])
print(mf['burst_familias'])

In [3]:
from ventanas_mejoras import poisson_burst_test, message_families
d = df_system.copy()
d['timestamp'] = pd.to_datetime(d['@timestamp'], utc=True)
d['msg_family'] = message_families(d)
for col in ['msg_family', 'sap_function_log_type', 'http_status_code', 'client_ip',
            'heathers_request_path', 'sap_function_application', 'sap_source_type',
            'region_code', 'headers_http_request_method', 'sap_app_env']:
    print(poisson_burst_test(d, col, top=5).to_string(), '\n')

      columna                                                                     categoria                    bucket  observado  esperado  neg_log10_p  umbral_bonferroni  esperado_sin_rafagas  significativo
0  msg_family       Thread contention detected — <N>% of threads blocked in <APP> on <SRC>. 2026-04-17 07:40:00+00:00         13      2.27         6.08               7.74                  5.74          False
1  msg_family      Same user logged in from two locations on <APP> — common with VPN usage. 2026-04-17 03:40:00+00:00          7      0.56         5.68               7.74                  5.74          False
2  msg_family  Cache hit ratio dropped to <N>% for <APP> — increased backend load on <SRC>. 2026-04-14 16:15:00+00:00         12      2.12         5.60               7.74                  5.74          False
3  msg_family                              Dependency <SRC> responded within SLA for <APP>. 2026-04-15 07:40:00+00:00         11      1.82         5.47               7.

In [4]:
pd.crosstab(d['msg_family'], d['sap_function_log_type']).to_csv('familia_x_log_type.csv')

## Convertir a CSV

In [ ]:
df_batch_system['window_index'] = range(1, len(df_batch_system) + 1)
df_batch_system.to_csv('../data/ventanas_separadas/batch_system_v3.csv', index=True, float_format='%.6f')

# LLM

In [ ]:
df_llm.columns

In [ ]:
lista_fila = df_llm.iloc[0].tolist()
display(lista_fila)

Posibles columnas para las ventanas de 5 minutos:
- total_llm_requests (Conteo total de logs)
- avg_llm_cost_usd (Promedio de los costos de las llamadas)
- total_llm_cost_usd (Suma de los costos de las llamadas)
- avg_token_ratio (llm_completion_tokens / llm_prompt_tokens. Para comparar el tamaño de los prompts de entrada con los de salida)
- avg_response_time_ms (Promedio de lo que tarda en responder el llm. Obtenido de llm_response_time_ms)

--

- llm_error_rate (Porcentaje de peticiones con llm_status = "error")
- llm_timeout_rate (Porcentaje de peticiones con llm_status = "timeout")

--

- temperature_variance (Qué tan dispersas están las temperaturas de las llamadas. Obtenido de llm_temperature)
- prompt_category_entropy (Entropía de Shannon normalizada entre 0 y 1; 1 indica categorías igualmente frecuentes. Obtenido de llm_prompt_category)
- stream_ratio (Porcentaje de llamadas en las que llm_stream = True)

--

- avg_prompt_length (Longitud promedio de los prompts. Obtenido de llm_prompt)
- max_prompt_length (Longitud del prompt más largo. Obtenido de llm_prompt)


Además de estas variables, estaría interesante agregar algo como "count_math_sus_words" donde tengamos una lista de palabras sospechosas y se contara cuántas veces aparecieron esas palabras en los prompts (como "bypass", "ignore al previous instructions", "give me the password"...)

In [ ]:
window_size = pd.Timedelta(minutes=5)
window_step = pd.Timedelta(minutes=2.5)

required_columns = [
    '@timestamp',
    'llm_cost_usd',
    'llm_completion_tokens',
    'llm_prompt_tokens',
    'llm_response_time_ms',
    'llm_status',
    'llm_temperature',
    'llm_prompt_category',
    'llm_stream',
    'llm_prompt',
]
logs = df_llm[required_columns].copy()
logs['timestamp'] = pd.to_datetime(logs['@timestamp'], utc=True, errors='coerce')

if logs['timestamp'].isna().any():
    invalid_count = logs['timestamp'].isna().sum()
    raise ValueError(f'Hay {invalid_count} logs sin un @timestamp válido.')

numeric_columns = [
    'llm_cost_usd',
    'llm_completion_tokens',
    'llm_prompt_tokens',
    'llm_response_time_ms',
    'llm_temperature',
]
for column in numeric_columns:
    logs[column] = pd.to_numeric(logs[column], errors='coerce')

logs['status'] = logs['llm_status'].astype('string').str.upper()
logs['token_ratio'] = logs['llm_completion_tokens'].div(
    logs['llm_prompt_tokens'].replace(0, np.nan)
)
logs['prompt_length'] = logs['llm_prompt'].astype('string').str.len()
logs['is_error'] = logs['status'].eq('ERROR').fillna(False)
logs['is_timeout'] = logs['status'].eq('TIMEOUT').fillna(False)
logs['is_stream'] = (
    logs['llm_stream'].astype('string').str.lower().eq('true').fillna(False)
)

# Cada solicitud se asigna a dos ventanas de 5 minutos separadas por 2.5 minutos.
window_starts = logs['timestamp'].dt.floor(window_step)
logs['_event_id'] = np.arange(len(logs))
events = pd.concat(
    [
        logs.assign(window_start=window_starts),
        logs.assign(window_start=window_starts - window_size + window_step),
    ],
    ignore_index=True,
)

# Si el último tramo observable dura menos de la mitad de una ventana,
# se integra a la ventana anterior para evitar una muestra desbalanceada.
last_window_start = events['window_start'].max()
if logs['timestamp'].max() - last_window_start < window_step:
    events.loc[events['window_start'].eq(last_window_start), 'window_start'] = (
        last_window_start - window_step
    )
    events = events.drop_duplicates(subset=['window_start', '_event_id'])

grouped = events.groupby('window_start', sort=True)
df_batch_llm = grouped.agg(
    total_llm_requests=('timestamp', 'size'),
    avg_llm_cost_usd=('llm_cost_usd', 'mean'),
    total_llm_cost_usd=('llm_cost_usd', 'sum'),
    avg_token_ratio=('token_ratio', 'mean'),
    avg_response_time_ms=('llm_response_time_ms', 'mean'),
    llm_error_rate=('is_error', 'mean'),
    llm_timeout_rate=('is_timeout', 'mean'),
    temperature_variance=('llm_temperature', 'var'),
    stream_ratio=('is_stream', 'mean'),
    avg_prompt_length=('prompt_length', 'mean'),
    max_prompt_length=('prompt_length', 'max'),
)

# Normalized Shannon entropy is 0 for one category and 1 for a uniform distribution.
category_counts = (
    events.dropna(subset=['llm_prompt_category'])
    .groupby(['window_start', 'llm_prompt_category'], observed=True)
    .size()
)
category_probabilities = (
    category_counts / category_counts.groupby(level=0).transform('sum')
)
category_entropy = -(
    category_probabilities * np.log2(category_probabilities)
).groupby(level=0).sum()
categories_per_window = category_counts.groupby(level=0).size()
normalized_category_entropy = category_entropy.div(
    np.log2(categories_per_window)
).where(categories_per_window > 1, 0)
df_batch_llm['prompt_category_entropy'] = (
    normalized_category_entropy.reindex(df_batch_llm.index).fillna(0)
)

df_batch_llm.index.name = 'window_start'
df_batch_llm = df_batch_llm[
    [
        'total_llm_requests',
        'avg_llm_cost_usd',
        'total_llm_cost_usd',
        'avg_token_ratio',
        'avg_response_time_ms',
        'llm_error_rate',
        'llm_timeout_rate',
        'temperature_variance',
        'prompt_category_entropy',
        'stream_ratio',
        'avg_prompt_length',
        'max_prompt_length',
    ]
]

display(df_batch_llm.head())

## Convertir a CSV

In [ ]:
df_batch_llm['window_index'] = range(1, len(df_batch_llm) + 1)
df_batch_llm.to_csv('../data/ventanas_separadas/batch_llm_indexed.csv', index=True, float_format='%.6f')